In [ ]:
# Cell 1 — auth + connect to BigQuery
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

project_id = "datathon-2026-506819"   # same project the team's tables live in
client = bigquery.Client(project=project_id)
print("Connected to BigQuery")
# Cell 1 — auth + connect to BigQuery
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

project_id = "datathon-2026-506819"   # same project the team's tables live in
client = bigquery.Client(project=project_id)
print("Connected to BigQuery")


Connected to BigQuery
Connected to BigQuery


In [ ]:

# Cell 2 — install pycountry (used later to filter out regional/income-group rows)
!pip install pycountry -q
import pycountry


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 49.1 MB/s eta 0:00:00


In [ ]:
# Cell 3 — upload the new dataset
# From the zip, use the "..._Data.csv" file (NOT "Series - Metadata.csv" --
# that one has the same data plus an indicator-definition block glued on
# at the bottom with a different column layout)
from google.colab import files
import pandas as pd
import re

uploaded = files.upload()   # picker opens -- sele
csv_path = list(uploaded.keys())[0]
print("Loaded:", csv_path)

raw = pd.read_csv(csv_path, encoding="latin1")
()

Saving b9371ef3-51b1-4c4a-9b10-6c3999c65be1_Data.csv to b9371ef3-51b1-4c4a-9b10-6c3999c65be1_Data.csv
Loaded: b9371ef3-51b1-4c4a-9b10-6c3999c65be1_Data.csv


()

In [ ]:
# Cell 4 — drop footer rows before doing anything else
# Same junk pattern as the food prices file: blank rows + source/date lines
# at the bottom with no Country Code.
clean = raw[raw["Country Code"].notna()].reset_index(drop=True)
print("Dropped", len(raw) - len(clean), "footer/junk rows")

Dropped 5 footer/junk rows


In [ ]:
# Cell 5 — melt wide year-columns into long format
# Year columns look like "1990 [YR1990]" -- pull t
# instead of trying to make that a BigQuery column name (it starts with a digit,
# which BigQuery rejects outright).

year_cols = [c for c in clean.columns if re.match(r"^\d{4} \[YR\d{4}\]$", c)]

long_df = clean.melt(
    id_vars=["Series Name", "Series Code", "Country Name", "Country Code"],
    value_vars=year_cols,
    var_name="Year_Raw",
    value_name="Value"
)

long_df["Year"] = long_df["Year_Raw"].str.extract(r"^(\d{4})").astype(int)
long_df["Value"] = long_df["Value"].replace("..", pd.NA)
long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")

long_df = long_df.rename(columns={
    "Series Name": "Series_Name",
    "Series Code": "Series_Code",
    "Country Name": "Country_Name",
    "Country Code": "Country_Code",
})[["Series_Name", "Series_Code", "Country_Name", "Country_Code", "Year", "Value"]]

print("Shape after melt/clean:", long_df.shape)
print("Non-null values:", long_df["Value"].notna().sum(), "/", len(long_df))
long_df.head()

Shape after melt/clean: (3180, 6)
Non-null values: 1367 / 3180


,Series_Name,Series_Code,Country_Name,Country_Code,Year,Value
0,"Incidence of malaria (per 1,000 population at ...",SH.MLR.INCD.P3,Africa Eastern and Southern,AFE,1990,NaN
1,"Incidence of malaria (per 1,000 population at ...",SH.MLR.INCD.P3,Africa Western and Central,AFW,1990,NaN
2,"Incidence of malaria (per 1,000 population at ...",SH.MLR.INCD.P3,Arab World,ARB,1990,NaN
3,"Incidence of malaria (per 1,000 population at ...",SH.MLR.INCD.P3,Caribbean small states,CSS,1990,NaN
4,"Incidence of malaria (per 1,000 population at ...",SH.MLR.INCD.P3,Central Europe and the Baltics,CEB,1990,NaN


In [ ]:
# Cell 6 — drop regional/income-group aggregates, keep only real countries
# (World Bank extracts include rows like "East Asia & Pacific", "Low income",
# "World", etc. mixed in with actual countries -- same cleanup you did for
# the GDP table.)
valid_iso3 = {c.alpha_3 for c in pycountry.countries} | {"XKX"}  # XKX = Kosovo, not in pycountry

dropped = long_df.loc[~long_df["Country_Code"].isin(valid_iso3), "Country_Name"].unique()
print("Dropped as non-national:", sorted(dropped))

long_df = long_df[long_df["Country_Code"].isin(valid_iso3)].reset_index(drop=True)
print("Remaining rows:", len(long_df))

Dropped as non-national: ['Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'Caribbean small states', 'Central Europe and the Baltics', 'Channel Islands', 'Early-demographic dividend', 'East Asia & Pacific', 'East Asia & Pacific (IDA & IBRD countries)', 'East Asia & Pacific (excluding high income)', 'Euro area', 'Europe & Central Asia', 'Europe & Central Asia (IDA & IBRD countries)', 'Europe & Central Asia (excluding high income)', 'European Union', 'Heavily indebted poor countries (HIPC)', 'High income', 'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only', 'IDA total', 'Late-demographic dividend', 'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)', 'Latin America & the Caribbean (IDA & IBRD countries)', 'Least developed countries: UN classification', 'Low & middle income', 'Low income', 'Lower middle income', 'Middle East, North Africa, Afghanistan & Pakistan', 'Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)',

In [ ]:
# Cell 7 — sanity-check column names are BigQuery-safe
assert long_df.columns.is_unique, "Duplicate column names"
assert all(re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c) for c in long_df.columns), \
    "Invalid BigQuery column name found"


In [ ]:
# Cell 8 — load into BigQuery, in its own dataset
dataset_name = "Malaria_Indicators"
table_name = "Malaria_Incidence"

dataset_ref = f"{project_id}.{dataset_name}"
table_ref = f"{dataset_ref}.{table_name}"

client.create_dataset(dataset_ref, exists_ok=True)

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE", autodetect=True)
load_job = client.load_table_from_dataframe(long_df, table_ref, job_config=job_config)
load_job.result()

table = client.get_table(table_ref)
print(f"Loaded {table.num_rows} rows into {table_ref}")
for field in table.schema:
    print(f"  {field.name} --> {field.field_type}")

Loaded 2592 rows into datathon-2026-506819.Malaria_Indicators.Malaria_Incidence
  Series_Name --> STRING
  Series_Code --> STRING
  Country_Name --> STRING
  Country_Code --> STRING
  Year --> INTEGER
  Value --> FLOAT


In [ ]:
# Cell 9 — verify by querying it back
check = client.query(f"""
SELECT Country_Name, Country_Code, Year, Value
FROM `{table_ref}`
ORDER BY Country_Name, Year
LIMIT 10
""").to_dataframe()
display(check)

value_check = client.query(f"""
SELECT
  COUNTIF(Value IS NOT NULL) AS rows_with_value,
  COUNTIF(Value IS NULL) AS rows_missing_value,
  COUNT(*) AS total_rows
FROM `{table_ref}`
""").to_dataframe()
display(value_check)

,Country_Name,Country_Code,Year,Value
0,Afghanistan,AFG,1990,NaN
1,Afghanistan,AFG,2000,84.48
2,Afghanistan,AFG,2016,27.37
3,Afghanistan,AFG,2017,28.99
4,Afghanistan,AFG,2018,22.33
5,Afghanistan,AFG,2019,14.15
6,Afghanistan,AFG,2020,8.39
7,Afghanistan,AFG,2021,6.26
8,Afghanistan,AFG,2022,9.19
9,Afghanistan,AFG,2023,13.26


,rows_with_value,rows_missing_value,total_rows
0,1047,1545,2592
